# LeetCode #218: The Skyline Problem

https://leetcode.com/problems/the-skyline-problem/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Event Sweep + Max Heap ★ | O(n log n) | O(n) |
| Divide and Conquer | O(n log n) | O(n) |
| Brute Force | O(n × max_x) | O(max_x) |

## Understanding the Methods
### Event Sweep + Max Heap (Optimal)
Create events for building starts (left edge) and ends (right edge). Sort events by x-coordinate. Maintain a max-heap of active building heights. At each event, if the max height changes, record a skyline point. Use negative heights for starts to handle tie-breaking in sorting.

### Divide and Conquer
Split buildings in half, recursively compute skylines for each half, then merge the two skylines by sweeping left to right and taking the max height at each x.

### Brute Force
For every x-coordinate, compute the max height among all buildings covering that point. Very slow for large coordinate ranges.

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<int>> GetSkyline(int[][] buildings) {
        var events = new List<int[]>();
        foreach (var b in buildings) {
            events.Add(new[] { b[0], -b[2] });
            events.Add(new[] { b[1], b[2] });
        }
        events.Sort((a, b) => a[0] != b[0] ? a[0] - b[0] : a[1] - b[1]);

        var result = new List<IList<int>>();
        var heights = new SortedDictionary<int, int>(Comparer<int>.Create((a, b) => b - a));
        heights[0] = 1;
        int prevMax = 0;

        foreach (var e in events) {
            if (e[1] < 0) {
                int h = -e[1];
                heights[h] = heights.GetValueOrDefault(h) + 1;
            } else {
                int h = e[1];
                if (heights[h] == 1) heights.Remove(h);
                else heights[h]--;
            }
            int curMax = heights.First().Key;
            if (curMax != prevMax) {
                result.Add(new List<int> { e[0], curMax });
                prevMax = curMax;
            }
        }
        return result;
    }
}

### Python

In [ ]:
import heapq

class Solution:
    def getSkyline(self, buildings: list[list[int]]) -> list[list[int]]:
        events = []
        for l, r, h in buildings:
            events.append((l, -h, r))
            events.append((r, 0, 0))
        events.sort()

        result = [[0, 0]]
        heap = [(0, float('inf'))]  # (-height, end_x)

        for x, neg_h, r in events:
            if neg_h < 0:
                heapq.heappush(heap, (neg_h, r))
            while heap[0][1] <= x:
                heapq.heappop(heap)
            max_h = -heap[0][0]
            if max_h != result[-1][1]:
                result.append([x, max_h])

        return result[1:]

### Go

In [ ]:
import (
    "container/heap"
    "sort"
)

type MaxHeap [][]int
func (h MaxHeap) Len() int            { return len(h) }
func (h MaxHeap) Less(i, j int) bool   { return h[i][0] > h[j][0] }
func (h MaxHeap) Swap(i, j int)        { h[i], h[j] = h[j], h[i] }
func (h *MaxHeap) Push(x interface{})  { *h = append(*h, x.([]int)) }
func (h *MaxHeap) Pop() interface{} {
    old := *h
    n := len(old)
    x := old[n-1]
    *h = old[:n-1]
    return x
}

func getSkyline(buildings [][]int) [][]int {
    events := [][]int{}
    for _, b := range buildings {
        events = append(events, []int{b[0], -b[2], b[1]})
        events = append(events, []int{b[1], 0, 0})
    }
    sort.Slice(events, func(i, j int) bool {
        if events[i][0] != events[j][0] {
            return events[i][0] < events[j][0]
        }
        return events[i][1] < events[j][1]
    })

    result := [][]int{{0, 0}}
    h := &MaxHeap{{0, 1<<31}}
    heap.Init(h)

    for _, e := range events {
        x := e[0]
        if e[1] < 0 {
            heap.Push(h, []int{-e[1], e[2]})
        }
        for (*h)[0][1] <= x {
            heap.Pop(h)
        }
        maxH := (*h)[0][0]
        if maxH != result[len(result)-1][1] {
            result = append(result, []int{x, maxH})
        }
    }
    return result[1:]
}

### Rust

In [ ]:
use std::collections::BinaryHeap;

impl Solution {
    pub fn get_skyline(buildings: Vec<Vec<i32>>) -> Vec<Vec<i32>> {
        let mut events: Vec<(i32, i32, i32)> = Vec::new();
        for b in &buildings {
            events.push((b[0], -b[2], b[1]));
            events.push((b[1], 0, 0));
        }
        events.sort();

        let mut result: Vec<Vec<i32>> = vec![vec![0, 0]];
        let mut heap: BinaryHeap<(i32, i32)> = BinaryHeap::new();
        heap.push((0, i32::MAX));

        for (x, neg_h, r) in &events {
            if *neg_h < 0 {
                heap.push((-neg_h, *r));
            }
            while heap.peek().unwrap().1 <= *x {
                heap.pop();
            }
            let max_h = heap.peek().unwrap().0;
            if max_h != *result.last().unwrap().last().unwrap() {
                result.push(vec![*x, max_h]);
            }
        }
        result[1..].to_vec()
    }
}

## Example Scenarios

### 1. Overlapping Buildings
**Input:** `buildings = [[2,9,10],[3,7,15],[5,12,12]]`  
Heights overlap; the skyline follows the tallest at each x. **Output:** `[[2,10],[3,15],[7,12],[12,0]]`

### 2. Non-Overlapping Buildings
**Input:** `buildings = [[0,2,3],[5,7,3]]`  
No overlap, two separate rectangles. **Output:** `[[0,3],[2,0],[5,3],[7,0]]`

### 3. Single Building
**Input:** `buildings = [[1,5,10]]`  
One building produces two skyline points. **Output:** `[[1,10],[5,0]]`

### 4. Nested Building
**Input:** `buildings = [[1,10,5],[2,8,10]]`  
Inner building is taller. **Output:** `[[1,5],[2,10],[8,5],[10,0]]`

### 5. Same Height Adjacent
**Input:** `buildings = [[1,3,5],[3,5,5]]`  
Adjacent buildings with same height merge into one segment. **Output:** `[[1,5],[5,0]]`

*Infographic will be added in a future update.*